# 02c — Free-text: Unsupervised Training

Free-text mode, unsupervised calibration (quantile thresholds + expert weights).
See `02_risk_score_guide.ipynb` for full parameter documentation.

## Setup

Imports → connect → load credentials → open session. See guide for session lifecycle rules.

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
# Run once at the start of every session (fresh or resumed).
import json, sys, os


#!pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers


print("SDK imported successfully.")

In [ ]:
# ── Paths — edit once ────────────────────────────────────────────────────────
API_KEY                = "xxxx"   # ← get your api key
ARTIFACT_DIR_LABELLING = "artifacts/free_text/labelling/"   # from 01b_labelling_free_text.ipynb
with open(os.path.join(ARTIFACT_DIR_LABELLING, "run_classification.json"), "r") as f:
    labelling_output = json.load(f)
ZIP_PATH               = labelling_output["zip_path"]
ARTIFACT_DIR           = "artifacts/free_text/risk_score/"  # risk score artifacts land here

import os, json
from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

client = OpsRiskClient()
client.wait_until_ready()
print(f"Zip: {ZIP_PATH}")

In [ ]:
config  = SessionConfig(name="risk-score-run")
session = client.open_session(
    api_key      = API_KEY,
    config       = config,
    artifact_dir = ARTIFACT_DIR,
)
session.import_model(ZIP_PATH)
print("Session ready. Auth ID:", session.auth_id)


In [ ]:
# ── Kernel-restart recovery — paste values from above ────────────────────────
# AUTH_ID      = "paste-auth-id-here"
# ARTIFACT_DIR = "artifacts/free_text/risk_score/"
#
# session = client.get_session(AUTH_ID, artifact_dir=ARTIFACT_DIR)
# session.status()

## Step 9 — `configure_training`

Upload training log file and configure quantile calibration parameters.
Free-text: `rolling_feature_type` supports `'sum'` and `'flag'` only.

In [ ]:
TRAINING_FILE           = "sample_data/free_text_training.csv"
ASSET_ID_COLUMN         = "asset_id"
TIMESTAMP_COLUMN        = "date"
DATE_FORMAT             = "%d/%m/%Y"  # 'infer' auto-detects
ROLLING_WINDOW          = 7
ROLLING_FEATURE_TYPE    = "sum"       # free-text: 'sum'|'flag'  fixed-log: adds 'ewm'|'all'
QUANTILE_FOR_THRESHOLDS = 0.99        # top 1 % of training activity → elevated risk
TRAINING_SHEET          = None        # XLSX only; None for CSV

training_config_result = session.configure_training(
    file_path               = TRAINING_FILE,
    asset_id_column         = ASSET_ID_COLUMN,
    timestamp_column        = TIMESTAMP_COLUMN,
    date_format             = DATE_FORMAT,
    rolling_window          = ROLLING_WINDOW,
    rolling_feature_type    = ROLLING_FEATURE_TYPE,
    quantile_for_thresholds = QUANTILE_FOR_THRESHOLDS,
    sheet_name              = TRAINING_SHEET,
    # supervised=False is the default — no failures file needed
)
print(json.dumps(training_config_result, indent=2))


## Step 10 — `train_risk_model`

Run unsupervised training on the server (~1–10 min). Artifacts downloaded to `artifact_dir` automatically.

In [ ]:
training_result = session.train_risk_model(timeout=3600)
print(json.dumps({k: v for k, v in training_result.items() if 'path' not in k}, indent=2))


### Training feature visualisations

Sanity-check calibrated thresholds and rolling feature distributions.

In [ ]:
# Convenience shortcuts
thresholds_path   = training_result["thresholds_path"]
training_fe_path  = training_result["training_fe_path"]
logs_mapping_path = training_result.get("logs_mapping_path")  # fixed-log only

# 1. Bar chart: calibrated threshold vs mean and max of training distribution
#    Helps spot thresholds that are too low (near mean) or unreachably high (near max)
helpers.plot_calibration_thresholds(
    thresholds_path,
    training_fe_path,
    #systems=["refrigeration", "electrical", "airflow"],
    systems=None  # None = all
)


In [ ]:
# 2. Histogram grid: one panel per subsystem showing the risk-feature distribution
#    The red vertical line is the calibrated threshold
helpers.plot_training_feature_distributions(
    training_fe_path,
    thresholds_path,
    #systems=["refrigeration", "electrical", "airflow"],
    systems=None,  # None = all
)


In [ ]:
# 3. Time series of a subsystem's rolling risk feature for selected assets
#    The dashed horizontal line is the calibrated threshold
INSPECT_SYSTEM  = "refrigeration_circuit"
INSPECT_SUBSYS  = "refrigerant_charge"

helpers.plot_training_feature_over_time(
    training_fe_path,
    thresholds_path,
    asset_id = ['HVAC-01', 'HVAC-03', 'HVAC-07'],
    system   = INSPECT_SYSTEM,
    subsystem = INSPECT_SUBSYS,
)


In [ ]:
# 4. Three-panel pipeline view for one asset/subsystem:
#      Top:    risk feature over time + threshold
#      Middle: rolling features (free-text) or per-log-code rolling values (fixed-log)
#      Bottom: binary presence/absence of events
helpers.plot_subsystem_feature_pipeline(
    training_fe_path,
    thresholds_path,
    asset_id    = "HVAC-03",
    system      = INSPECT_SYSTEM,
    subsystem   = INSPECT_SUBSYS,
    is_free_text = training_config_result["is_free_text"],
    logs_mapping = logs_mapping_path,  # only used when is_free_text=False
)


## Step 11 — `configure_generation`

Upload log file to score. Uses expert weights from `model_config.json`.

In [ ]:
GENERATION_FILE  = "sample_data/free_text_generation.csv"
DATE_FORMAT      = "%d/%m/%Y"
GENERATION_SHEET = None

generation_config_result = session.configure_generation(
    file_path      = GENERATION_FILE,
    date_format    = DATE_FORMAT,
    sheet_name     = GENERATION_SHEET,
    weights_source = "labelling",  # expert weights from labelling step
)
print(json.dumps(generation_config_result, indent=2))


## Step 12 — `generate_risk_scores` 🔒 ONE-WAY DOOR

Generate per-asset risk scores. Downloads complete model zip to `OUTPUT_DIR`.

In [ ]:
# Run generation — ONE-WAY DOOR: cannot be repeated on this session.
# The complete model zip is downloaded to OUTPUT_DIR automatically.
OUTPUT_DIR = "outputs/free_text/"

generation_result = session.generate_risk_scores(OUTPUT_DIR, timeout=3600)

parquet_path = generation_result.get("parquet_path")
print(f"Risk scores parquet: {parquet_path}")
print(f"Assets scored:       {generation_result['assets_scored']}")
print(f"Date range:          {generation_result['date_range']}")
print("Session wiped automatically — use the zip in OUTPUT_DIR to restart.")


In [ ]:
# For unsupervised runs the weights always come from model_config.json
WEIGHTS_CONFIG = f"{ARTIFACT_DIR}/model_config.json"
print(f"Weights config: {WEIGHTS_CONFIG}")


### Risk score visualisations

Plot fleet-wide rankings, heatmaps, single-asset breakdowns, and log drill-downs.
`classified_logs.parquet` is available for free-text log drill-down.

In [ ]:
# Top-N assets ranked by aggregated operational risk
helpers.plot_asset_risk_ranking(
    parquet_path,
    aggregation = "max",   # "mean" | "max" | "min" | "median"
    top_n       = 20,
    title       = "Asset Operational Risk Scores",
)

# Distribution of risk scores across all records
helpers.plot_risk_score_distribution(parquet_path)


In [ ]:
# Multi-asset heatmap: rows = assets, columns = dates
# metric= accepts "operational_risk" or any "{system}_system_risk" column
print("Available metrics:")
for m in helpers.list_risk_metrics(parquet_path): print(f"  {m}")

helpers.plot_risk_heatmap(
    parquet_path,
    metric    = "operational_risk",
    asset_ids = ['HVAC-01', 'HVAC-03', 'HVAC-07'],  # None = all assets
    title     = "Multi-Asset Operational Risk Heatmap",
)


In [ ]:
# Tile panel for a single date — one operational-risk tile per asset
# with system-level tiles below.  Colour: blue (0–33), yellow (33–66), red (66–100)
SCORE_DATE = "2024-02-15"   # ← edit
helpers.plot_daily_risk_snapshot(
    parquet_path,
    date      = SCORE_DATE,
    min_score = 0.0,           # hide assets below this score
    asset_ids = ['HVAC-01', 'HVAC-03', 'HVAC-07'],
)


In [ ]:
# Single asset — heatmap of all risk types over time
ASSET_TO_INSPECT = "HVAC-07"   # ← edit

# Raw scores
helpers.plot_asset_risk_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    title=f"Risk Analysis — {ASSET_TO_INSPECT}",
)

# Weighted view (system rows scaled by their model weight)
helpers.plot_asset_risk_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    weighted=True, model_config=WEIGHTS_CONFIG,
    title=f"Risk Analysis — Weighted — {ASSET_TO_INSPECT}",
)


In [ ]:
# Stacked weighted system contributions + operational risk line
helpers.plot_system_contributions(
    parquet_path,
    asset_id     = ASSET_TO_INSPECT,
    model_config = WEIGHTS_CONFIG,
)


In [ ]:
# Drill into one system: subsystem heatmap over time
DRILL_SYSTEM = "refrigeration_circuit"   # ← edit

helpers.plot_asset_system_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    system=DRILL_SYSTEM, title=f"{DRILL_SYSTEM} — {ASSET_TO_INSPECT}",
)
helpers.plot_asset_system_over_time(
    parquet_path, asset_id=ASSET_TO_INSPECT,
    system=DRILL_SYSTEM, weighted=True, model_config=WEIGHTS_CONFIG,
    title=f"{DRILL_SYSTEM} — Weighted — {ASSET_TO_INSPECT}",
)


In [ ]:
# Log-level drill-down: rolling features + binary presence + log counts
# for a specific asset / system / subsystem window
#
# Free-text mode  → source = classified_logs.parquet
# Fixed-log mode  → source = risk_scores.parquet + logs_mapping
import os
ASSET_ID   = "HVAC-07"
SYSTEM     = "refrigeration_circuit"
SUBSYSTEM  = "refrigerant_charge"
END_DATE   = "2024-02-15"
DAYS_BACK  = 60
LOG_COLUMN = "maintenance_log"

classified_logs_path = generation_result.get("classified_logs_path")
logs_mapping_path    = generation_result.get("logs_mapping_path")

if classified_logs_path and os.path.exists(classified_logs_path):
    # Free-text mode
    helpers.plot_subsystem_log_activity(
        classified_logs_path,
        asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
        date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
        risk_source=parquet_path,
    )
elif parquet_path and logs_mapping_path:
    # Fixed-log mode
    helpers.plot_subsystem_log_activity(
        parquet_path,
        asset_id=ASSET_ID, system=SYSTEM, subsystem=SUBSYSTEM,
        date=END_DATE, days_back=DAYS_BACK, log_column=LOG_COLUMN,
        risk_source=parquet_path, logs_mapping=logs_mapping_path, is_free_text=False,
    )


In [ ]:
if not os.path.exists(classified_logs_path):
    print("classified_logs.parquet not available — this helper is for free-text mode only.")
    print(f"Expected at: {classified_logs_path}")
else:
    logs_df = helpers.get_logs_for_subsystem(
        classified_logs_path,
        asset_id=ASSET_ID,
        system=SYSTEM,
        subsystem=SUBSYSTEM,
        date=END_DATE,
        days_back=DAYS_BACK,
        log_column=LOG_COLUMN,
    )
    print(f"{len(logs_df)} log entries found")
    display(logs_df)